In [1]:
%pip install chronos-forecasting
%pip install ipywidgets
%pip install transformers accelerate


[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
  Using cached ipywidgets-8.1.8-py3-none-any.whl.metadata (2.4 kB)
  Using cached widgetsnbextension-4.0.15-py3-none-any.whl.metadata (1.6 kB)
  Using cached jupyterlab_widgets-3.0.16-py3-none-any.whl.metadata (20 kB)
Using cached ipywidgets-8.1.8-py3-none-any.whl (139 kB)
Using cached jupyterlab_widgets-3.0.16-py3-none-any.whl (914 kB)
Using cached widgetsnbextension-4.0.15-py3-none-any.whl (2.2 MB)

[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd
import numpy as np
import findspark
import pyspark as spark
from pyspark.sql import SparkSession
from chronos import Chronos2Pipeline
from pyspark.sql.types import StructType, StructField,FloatType,TimestampType,StringType
from pyspark.sql.functions import col,to_json,struct,from_json,to_timestamp,count, collect_list
from dotenv import load_dotenv
import os
from huggingface_hub import login
import torch

In [3]:
load_dotenv()

True

In [2]:
chronos2_pipeline = Chronos2Pipeline.from_pretrained("../Offline-Phase/chronos2_zero_shot")

### Pandas Functions from the offline phase

In [3]:
def extract_time_features(df, timestamp_col='timestamp'):


    df['hour_sin'] = np.sin(2 * np.pi * df[timestamp_col].dt.hour / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df[timestamp_col].dt.hour / 24)


    df['month_sin'] = np.sin(2 * np.pi * (df[timestamp_col].dt.month - 1) / 12)
    df['month_cos'] = np.cos(2 * np.pi * (df[timestamp_col].dt.month - 1) / 12)
    
    
    df['day_sin'] = np.sin(2 * np.pi * df[timestamp_col].dt.dayofweek / 7)
    df['day_cos'] = np.cos(2 * np.pi * df[timestamp_col].dt.dayofweek / 7)

    df['is_weekend'] = df[timestamp_col].dt.dayofweek.isin([5, 6]).astype(int)

 
    df['is_heating_season'] = df[timestamp_col].dt.month.isin([11, 12, 1, 2, 3]).astype(int)

    return df

In [4]:
def append_neighbors(df_hourly, neighbors_df, weather_cols=["humidity", "pressure","temperature", "wind_speed"], k_search=20, k_keep=3):
    # 1. Standardize the neighbor list
    # Ensure we only take the top K based on distance
    neighbors_topk = (
        neighbors_df.sort_values(["sensor_id", "distance_km"])
        .groupby("sensor_id")
        .head(k_search)
        .copy()
    )

    # Track original distance rank
    neighbors_topk['dist_rank'] = neighbors_topk.groupby("sensor_id").cumcount() + 1

    # 2. Merge with main data
    # We use 'neighbor_id' from the matrix to match 'sensorId' in the hourly data
    neighbor_values = neighbors_topk.merge(
        df_hourly[['sensorId', 'timestamp'] + weather_cols],
        left_on='neighbor_id',
        right_on='sensorId',
        how='inner'
    )

    # 3. Filter for availability
    # The 'sensor_id' here is the ORIGINAL sensor we are finding neighbors for
    available_topk = (
        neighbor_values.sort_values(['sensor_id', 'timestamp', 'dist_rank'])
        .groupby(['sensor_id', 'timestamp'])
        .head(k_keep)
        .copy()
    )

    # Create the 1, 2, 3 rank for the wide-format columns
    available_topk['final_rank'] = available_topk.groupby(['sensor_id', 'timestamp']).cumcount() + 1

    # 4. Pivot to wide format
    pivot_df = available_topk.pivot(
        index=['sensor_id', 'timestamp'],
        columns='final_rank',
        values=weather_cols
    )

    # Clean up column names: neighbor1_temp, neighbor2_temp, etc.
    if isinstance(pivot_df.columns, pd.MultiIndex):
        pivot_df.columns = [f"neighbor{rank}_{col}" for col, rank in pivot_df.columns]
    else:
        # Handle case with only one weather column
        pivot_df.columns = [f"neighbor{i}_{weather_cols[0]}" for i in pivot_df.columns]

    pivot_df = pivot_df.reset_index()

    # 5. Final Join back to original data
    df_result = df_hourly.merge(
        pivot_df,
        left_on=['sensorId', 'timestamp'],
        right_on=['sensor_id', 'timestamp'],
        how='left'
    ).drop(columns=['sensor_id'])

    return df_result

In [5]:
neighbourhood_matrix = pd.read_csv("../data/neighbors_data/bitola_sensor_distances.csv")
neighbourhood_matrix

,sensor_id,neighbor_id,distance_km
0,d241a044-0a06-40c2-9d90-c91fd0a95060,fec52a19-9148-4350-a1b4-ae0da05ee199,7.082000
1,fec52a19-9148-4350-a1b4-ae0da05ee199,d241a044-0a06-40c2-9d90-c91fd0a95060,7.082000
2,d241a044-0a06-40c2-9d90-c91fd0a95060,be427cee-4c3a-4aa2-a1ce-9795a74533be,8.838588
3,be427cee-4c3a-4aa2-a1ce-9795a74533be,d241a044-0a06-40c2-9d90-c91fd0a95060,8.838588
4,d241a044-0a06-40c2-9d90-c91fd0a95060,c3f3da9b-9fd3-4037-94d3-598d655e6be9,10.004966
...,...,...,...
457,7b316592-8036-41e2-b8dc-b06b6a9afd54,40f081a6-4095-43f7-bffb-64e2af8c026e,1.049671
458,40f081a6-4095-43f7-bffb-64e2af8c026e,692c454c-a1ad-41fa-b3ca-aa1cb7d55d30,3.044465
459,692c454c-a1ad-41fa-b3ca-aa1cb7d55d30,40f081a6-4095-43f7-bffb-64e2af8c026e,3.044465
460,7b316592-8036-41e2-b8dc-b06b6a9afd54,692c454c-a1ad-41fa-b3ca-aa1cb7d55d30,2.083640


In [6]:
def load_context():
    context_df = pd.read_csv("Context_bitola.csv")
    context_df = context_df.drop(columns=['city'])
    context_df['timestamp'] = pd.to_datetime(context_df['timestamp'],utc=True)
    return context_df

In [7]:
def load_forecast():
    forecast_df = pd.read_csv('../data/raw/bitola_forecast_weather.csv')
    forecast_df['timestamp'] = pd.to_datetime(forecast_df['timestamp'],utc=True)
    forecast_df = extract_time_features(forecast_df)
    forecast_df = append_neighbors(forecast_df, neighbourhood_matrix)
    return forecast_df

In [8]:
def build_future_df(ts_df, forecast_df):

    ts_df = ts_df.copy()
    forecast_df = forecast_df.copy()

    ts_df["timestamp"] = pd.to_datetime(ts_df["timestamp"], utc=True)
    forecast_df["timestamp"] = pd.to_datetime(forecast_df["timestamp"], utc=True)

    future_rows = []

    for sensor_id in ts_df["sensorId"].unique():

        sensor_current = ts_df[ts_df["sensorId"] == sensor_id]

        base_time = sensor_current["timestamp"].max()

        future_times = [
            base_time + pd.Timedelta(hours=i)
            for i in range(1, 25)
        ]

        sensor_future = pd.DataFrame({
            "sensorId": sensor_id,
            "timestamp": future_times
        })

        sensor_forecast = forecast_df[
            forecast_df["sensorId"] == sensor_id
        ]

        sensor_future = sensor_future.merge(
            sensor_forecast,
            on=["sensorId", "timestamp"],
            how="left"
        )

        future_rows.append(sensor_future)

    future_df = pd.concat(future_rows, ignore_index=True)

    return future_df

In [23]:
def predict_target(current_df, context_df, target, pipeline, ID_COL, TIME_COL):
    context_target = context_df.copy()
   
    forecast_df = pipeline.predict_df(
        df=context_target,
        prediction_length=24,
        target=target,
        id_column=ID_COL,
        future_df=current_df,
        validate_inputs=False
    )

    forecast_df = forecast_df.sort_values([ID_COL, TIME_COL])


    current_pred = (
        forecast_df
        .groupby(ID_COL)
        .first()
        .reset_index()
        .rename(columns={"predictions": target})
    )


    forecast_24h = (
        forecast_df
        .groupby(ID_COL)["predictions"]
        .apply(lambda x: x.head(24).tolist())
        .reset_index()
        .rename(columns={"predictions": f"{target}_forecast_24h"})
    )

  
    result_df = current_df.merge(
        current_pred[[ID_COL, target]],
        on=ID_COL,
        how="left"
    )


    result_df = result_df.merge(
        forecast_24h,
        on=ID_COL,
        how="left"
    )

    return result_df

In [10]:
def process_batch(current_df, context_df):
    ID_COL = "sensorId"
    TIME_COL = "timestamp"

    current_df[TIME_COL] = pd.to_datetime(current_df[TIME_COL], utc=True)
    context_df[TIME_COL] = pd.to_datetime(context_df[TIME_COL], utc=True)

    context_sorted = context_df.copy().sort_values([ID_COL, TIME_COL])

    print("Context max timestamp:", context_sorted[TIME_COL].max())

    pm10_df = predict_target(
        current_df,
        context_sorted,
        target="pm10",
        pipeline=chronos2_pipeline,
        ID_COL=ID_COL,
        TIME_COL=TIME_COL,
    )

    pm25_df = predict_target(
        current_df,
        context_sorted,
        target="pm25",
        pipeline=chronos2_pipeline,
        ID_COL=ID_COL,
        TIME_COL=TIME_COL,
    )

    return pm10_df, pm25_df

In [11]:
def write_to_kafka(df, topic):
    spark_df = spark.createDataFrame(df)

    kafka_df = spark_df.select(
        col("sensorId").cast("string").alias("key"),
        to_json(struct(*spark_df.columns)).alias("value")
    )

    kafka_df.write \
        .format("kafka") \
        .option("kafka.bootstrap.servers", "localhost:9092") \
        .option("topic", topic) \
        .save()

In [12]:
context_df = load_context()
forecast_df = load_forecast()

def foreach_batch(batch_df, epoch_id):
    global context_df

    print(f"\nBatch received! Epoch: {epoch_id}")

    if batch_df.rdd.isEmpty():
        return

    current_df = batch_df.toPandas()

    if current_df.empty:
        print("Empty batch — skipping")
        return

    current_df = current_df.sort_values("timestamp")

    for _, row in current_df.iterrows():
        ts = row["timestamp"]

        ts_df = pd.DataFrame([r.asDict() for r in row["rows"]])

        print(f"\nProcessing timestamp: {ts}")
        print(f"Sensor count: {len(ts_df)}")

        context_max_ts = pd.to_datetime(context_df["timestamp"], utc=True).max() if not context_df.empty else None
        if context_max_ts is not None and pd.to_datetime(ts, utc=True) <= context_max_ts:
            print(f"Skipping timestamp {ts} because it is not after context max {context_max_ts}")
            continue
        
        incoming_ids = set(ts_df['sensorId'].unique())
        existing_ids = set(context_df['sensorId'].unique()) if not context_df.empty else set()
        new_ids = incoming_ids - existing_ids

        if new_ids:
            print(f"New sensors detected: {new_ids}")

            numeric_columns = [
                col for col in context_df.columns
                if col in ['temperature','wind_speed','humidity','pm10','pm25','pressure']
            ]

            city_baseline = context_df.groupby('timestamp')[numeric_columns].median().reset_index()

            proxy_rows = []

            for sid in new_ids:
                proxy_history = city_baseline.copy()
                proxy_history['sensorId'] = sid

                temp_combined = pd.concat([context_df, proxy_history], ignore_index=True)

                refined_data = append_neighbors(temp_combined, neighbourhood_matrix)
                refined_data = extract_time_features(refined_data)

                new_sensor_proxy = refined_data[refined_data['sensorId'] == sid]
                proxy_rows.append(new_sensor_proxy)

            context_df = pd.concat([context_df, *proxy_rows], ignore_index=True)

        ts_df["timestamp"] = pd.to_datetime(ts_df["timestamp"], utc=True)

        ts_df = extract_time_features(ts_df)
        ts_df = append_neighbors(ts_df, neighbourhood_matrix)

        future_df = build_future_df(ts_df,forecast_df)

        pm10_df, pm25_df = process_batch(future_df, context_df)

        if pm10_df is None or pm25_df is None:
            print("Prediction skipped")
            continue

        write_to_kafka(pm10_df, topic="FullPm10WeatherData_ZeroShot")
        write_to_kafka(pm25_df, topic="FullPm25WeatherData_ZeroShot")
        
        ts_df = ts_df.merge(
            pm10_df[['sensorId', 'timestamp', 'pm10']],
            on=['sensorId', 'timestamp'],
            how='left'
        )

        ts_df = ts_df.merge(
            pm25_df[['sensorId', 'timestamp', 'pm25']],
            on=['sensorId', 'timestamp'],
            how='left'
        )
        context_df = pd.concat([context_df, ts_df], ignore_index=True)

        context_df = (
            context_df
            .sort_values(["sensorId", "timestamp"])
            .drop_duplicates(subset=["sensorId", "timestamp"], keep="last")
            .groupby("sensorId")
            .tail(72)
            .reset_index(drop=True)
        )

        print(f"Prediction done for {ts}")
        print(f"Context size: {len(context_df)}")

# Online Phase (Main Program)

In [13]:
findspark.init()

In [14]:
spark = SparkSession.builder \
    .appName("KafkaConsumerExample") \
    .config( "spark.jars.packages","org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.7")  \
    .getOrCreate()

:: loading settings :: url = jar:file:/Users/jovan/Documents/Kodovi/RNMP/Project/venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/jovan/.ivy2/cache
The jars for the packages stored in: /Users/jovan/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-6f26c340-72f4-4592-aeb4-13ed65b98c6b;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.7 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.7 in central
	found org.apache.kafka#kafka-clients;3.4.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.5 in central
	found org.slf4j#slf4j-api;2.0.7 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.apache.commons#commons-pool2;2.11.1 in central
:: resolution report :: resolve 164ms :: artifacts dl 4ms
	:: 

In [15]:
df = spark \
    .readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "localhost:9092") \
    .option("subscribePattern", "sensor_.*") \
    .load()

In [16]:
df.printSchema()

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



In [17]:
schema = StructType([
    StructField("timestamp",TimestampType(),True),
    StructField("sensorId",StringType(),True),
    StructField("lat",FloatType(),True),
    StructField("lon",FloatType(),True),
    StructField("humidity",FloatType(),True),
    StructField("pressure",FloatType(),True),
    StructField("temperature",FloatType(),True),
    StructField("wind_speed",FloatType(),True)
])

In [18]:
parsed_df = df.selectExpr("CAST(value AS STRING)") \
    .select(from_json(col("value"), schema).alias("data")) \
    .select("data.*") \
    .drop("lat", "lon")

In [19]:
parsed_df = parsed_df.withColumn(
    "timestamp",
    to_timestamp("timestamp")
)

In [20]:
parsed_df.printSchema()

root
 |-- timestamp: timestamp (nullable = true)
 |-- sensorId: string (nullable = true)
 |-- humidity: float (nullable = true)
 |-- pressure: float (nullable = true)
 |-- temperature: float (nullable = true)
 |-- wind_speed: float (nullable = true)



In [21]:
grouped_df = parsed_df \
    .withWatermark("timestamp", "5 minutes") \
    .groupBy("timestamp") \
    .agg(
        collect_list(struct("*")).alias("rows"),
        count("*").alias("sensor_count")
    )

In [ ]:
query = grouped_df.writeStream \
    .foreachBatch(foreach_batch) \
    .start()

query.awaitTermination()

26/05/10 18:48:46 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /private/var/folders/18/8m8jfl3d72v4p728z2p8fn8c0000gn/T/temporary-c9c95361-0aff-42aa-a632-7879f5e41075. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/05/10 18:48:46 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/05/10 18:48:46 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.



Batch received! Epoch: 0



Batch received! Epoch: 1



Batch received! Epoch: 2


26/05/10 18:49:04 WARN InternalKafkaConsumerPool: Pool exceeds its soft max size, cleaning up idle objects...
26/05/10 18:49:04 WARN InternalKafkaConsumerPool: Pool exceeds its soft max size, cleaning up idle objects...


...


Processing timestamp: 2025-12-02 07:00:00
Sensor count: 22
Context max timestamp: 2025-12-02 06:00:00+00:00
Prediction done for 2025-12-02 07:00:00
Context size: 1584

Processing timestamp: 2025-12-02 08:00:00
Sensor count: 22
Context max timestamp: 2025-12-02 07:00:00+00:00
Prediction done for 2025-12-02 08:00:00
Context size: 1584

Processing timestamp: 2025-12-02 09:00:00
Sensor count: 22
Context max timestamp: 2025-12-02 08:00:00+00:00
Prediction done for 2025-12-02 09:00:00
Context size: 1584

Batch received! Epoch: 11



Processing timestamp: 2025-12-02 10:00:00
Sensor count: 22
Context max timestamp: 2025-12-02 09:00:00+00:00
Prediction done for 2025-12-02 10:00:00
Context size: 1584

Processing timestamp: 2025-12-02 11:00:00
Sensor count: 22
Context max timestamp: 2025-12-02 10:00:00+00:00
Prediction done for 2025-12-02 11:00:00
Context size: 1584

Processing timestamp: 2025-12-02 12:00:00
Sensor count: 22
Context max timestamp: 2025-12-02 11:00:00+00:00
Prediction done for 2025-12-02 12:00:00
Context size: 1584

Processing timestamp: 2025-12-02 13:00:00
Sensor count: 22
Context max timestamp: 2025-12-02 12:00:00+00:00
Prediction done for 2025-12-02 13:00:00
Context size: 1584

Processing timestamp: 2025-12-02 14:00:00
Sensor count: 22
Context max timestamp: 2025-12-02 13:00:00+00:00
Prediction done for 2025-12-02 14:00:00
Context size: 1584

Processing timestamp: 2025-12-02 15:00:00
Sensor count: 22
Context max timestamp: 2025-12-02 14:00:00+00:00
Prediction done for 2025-12-02 15:00:00
Context siz

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/Users/jovan/Documents/Kodovi/RNMP/Project/venv/lib/python3.11/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/jovan/Documents/Kodovi/RNMP/Project/venv/lib/python3.11/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/socket.py", line 706, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

26/05/10 18:50:49 ERROR Executor: Exception in task 8.0 in stage 336.0 (TID 8237)
org.apache.spark.api.python.PythonException: Traceback (most recent call last):
  File "/Users/jovan/Documents/Kodovi/RNMP/Project/venv/lib/python3.11/site-packages/pyspark/python/lib/pyspark.zip/pyspark/worker.py", line 1094, in main
    split_index = read_int(infile)
                  ^^^^^^^^^^^^^^^^
  File "/Users/jovan/Documents/Kodovi/RNMP/Project/venv/lib/python3.11/site-packages/pyspark/python/lib/pyspark.zip/pyspark/serializers.py", line 594, in read_int
    length = stream.read(4)
             ^^^^^^^^^^^^^^
KeyboardInterrupt

	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator.handlePythonException(PythonRunner.scala:572)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scala:784)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scala:766)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator.hasNext(PythonRunner.scala:525)
	at o